<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning in credit scoring <br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> Classification</font>
  </div>

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 20px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Arystan, Sirine, Jules and Augustin
      
  </div>

# 1. Imporiting Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier


from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import optuna

pd.set_option("display.max_columns", None)

/opt/miniconda3/envs/arystan_credit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Data Loading

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
df = pd.read_csv(path_to_data / 'preprocessed_data.csv')

In [4]:
y = df["bad"]
X = df.drop(columns=["bad", "ID"])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# RANDOM FOREST CLASSIFIER

In [6]:
rf_base = RandomForestClassifier(random_state=42)
%time rf_base.fit(X_train, y_train)

CPU times: user 969 ms, sys: 13.3 ms, total: 983 ms
Wall time: 992 ms


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
y_pred_default = rf_base.predict(X_test)

In [8]:
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



In [9]:
rf_default_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')
%time rf_default_balanced.fit(X_train, y_train)

CPU times: user 997 ms, sys: 12.1 ms, total: 1.01 s
Wall time: 1.01 s


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
y_pred_default_balanced = rf_default_balanced.predict(X_test)
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



In [11]:
feature_importance = pd.DataFrame({'feature': X.columns,'importance': rf_default_balanced.feature_importances_}).sort_values('importance', ascending=False)
feature_importance.head(20)

,feature,importance
13,AGE,0.220440
15,LOG_INCOME,0.208378
14,EXPERIENCE,0.142290
6,NAME_FAMILY_STATUS,0.050596
4,NAME_INCOME_TYPE,0.050519
12,CNT_FAM_MEMBERS,0.048632
5,NAME_EDUCATION_TYPE,0.044144
1,FLAG_OWN_CAR,0.035269
3,CNT_CHILDREN,0.034801
10,FLAG_PHONE,0.034329


In [12]:
rf_default_balanced_subsample = RandomForestClassifier(random_state=42,
                                    class_weight='balanced_subsample')
%time rf_default_balanced_subsample.fit(X_train, y_train)

CPU times: user 1.3 s, sys: 38.8 ms, total: 1.34 s
Wall time: 1.48 s


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
y_pred_default_balanced_subsample = rf_default_balanced_subsample.predict(X_test)
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



## RF + Hyperparameter Tuning

In [14]:
def objective_rf(trial, X_train, y_train, X_val, y_val):

    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 15)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    class_weight = trial.suggest_categorical("class_weight",
                                             ["balanced", "balanced_subsample"])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)

    return f1

In [15]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [ ]:
study_rf = optuna.create_study(
    study_name="rf_opt",
    direction="maximize",
    storage="sqlite:///.../models/rf_opt.db",
    load_if_exists=True
)

study_rf.optimize(lambda trial: objective_rf(trial, X_train, y_train, X_val, y_val),n_trials=50, show_progress_bar=True)

[I 2025-12-07 15:19:33,295] A new study created in RDB with name: rf_opt
Best trial: 0. Best value: 0.253275:   2%|▏         | 1/50 [00:00<00:25,  1.92it/s]

[I 2025-12-07 15:19:33,819] Trial 0 finished with value: 0.25327510917030566 and parameters: {'n_estimators': 203, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.25327510917030566.


Best trial: 1. Best value: 0.260163:   4%|▍         | 2/50 [00:01<00:36,  1.31it/s]

[I 2025-12-07 15:19:34,749] Trial 1 finished with value: 0.2601626016260163 and parameters: {'n_estimators': 247, 'max_depth': 25, 'min_samples_split': 14, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample'}. Best is trial 1 with value: 0.2601626016260163.


Best trial: 1. Best value: 0.260163:   6%|▌         | 3/50 [00:01<00:28,  1.64it/s]

...

[I 2025-12-07 15:20:12,614] Trial 47 finished with value: 0.25101214574898784 and parameters: {'n_estimators': 173, 'max_depth': 26, 'min_samples_split': 7, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}. Best is trial 33 with value: 0.2627118644067797.


Best trial: 33. Best value: 0.262712:  98%|█████████▊| 49/50 [00:39<00:00,  1.54it/s]

[I 2025-12-07 15:20:13,161] Trial 48 finished with value: 0.25 and parameters: {'n_estimators': 259, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 3, 'class_weight': 'balanced'}. Best is trial 33 with value: 0.2627118644067797.


Best trial: 33. Best value: 0.262712: 100%|██████████| 50/50 [00:40<00:00,  1.23it/s]

[I 2025-12-07 15:20:13,949] Trial 49 finished with value: 0.2575107296137339 and parameters: {'n_estimators': 218, 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 1, 'class_weight': 'balanced_subsample'}. Best is trial 33 with value: 0.2627118644067797.


In [17]:
print("Best F1:", study_rf.best_value)
print("Best params:", study_rf.best_params)

Best F1: 0.2627118644067797
Best params: {'n_estimators': 239, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 3, 'class_weight': 'balanced'}


In [18]:
best_params = study_rf.best_params

best_rf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

y_pred_best = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      5376
           1       0.17      0.35      0.23        93

    accuracy                           0.96      5469
   macro avg       0.58      0.66      0.61      5469
weighted avg       0.97      0.96      0.97      5469



## RF + SMOTE + Hyperparameter Tuning

In [19]:
def objective_rf_smote(trial, X_train, y_train, X_val, y_val):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    class_weight = trial.suggest_categorical("class_weight",
                                             ["balanced", "balanced_subsample"])

    smote = SMOTE(sampling_strategy=0.3, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_res, y_train_res)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)

    return f1


In [ ]:
study_rf_smote = optuna.create_study(
    study_name="rf_smote_opt",
    direction="maximize",
    storage="sqlite:///.../models/rf_smote.db",
    load_if_exists=True
)

[I 2025-12-07 15:20:14,648] A new study created in RDB with name: rf_smote_opt


In [21]:
study_rf_smote.optimize(lambda trial: objective_rf_smote(trial, X_train, y_train, X_val, y_val), n_trials=50, show_progress_bar=True)

Best trial: 0. Best value: 0.163842:   2%|▏         | 1/50 [00:00<00:30,  1.62it/s]

[I 2025-12-07 15:20:15,283] Trial 0 finished with value: 0.1638418079096045 and parameters: {'n_estimators': 110, 'max_depth': 24, 'min_samples_split': 16, 'min_samples_leaf': 6, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.1638418079096045.


Best trial: 0. Best value: 0.163842:   4%|▍         | 2/50 [00:01<00:28,  1.66it/s]

[I 2025-12-07 15:20:15,883] Trial 1 finished with value: 0.145 and parameters: {'n_estimators': 198, 'max_depth': 27, 'min_samples_split': 17, 'min_samples_leaf': 9, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.1638418079096045.


Best trial: 0. Best value: 0.163842:   6%|▌         | 3/50 [00:01<00:23,  1.99it/s]

...

[I 2025-12-07 15:20:51,875] Trial 47 finished with value: 0.2140221402214022 and parameters: {'n_estimators': 187, 'max_depth': 29, 'min_samples_split': 7, 'min_samples_leaf': 4, 'class_weight': 'balanced'}. Best is trial 39 with value: 0.2631578947368421.


Best trial: 39. Best value: 0.263158:  98%|█████████▊| 49/50 [00:37<00:00,  1.81it/s]

[I 2025-12-07 15:20:52,357] Trial 48 finished with value: 0.1377672209026128 and parameters: {'n_estimators': 163, 'max_depth': 27, 'min_samples_split': 5, 'min_samples_leaf': 10, 'class_weight': 'balanced'}. Best is trial 39 with value: 0.2631578947368421.


Best trial: 39. Best value: 0.263158: 100%|██████████| 50/50 [00:38<00:00,  1.30it/s]

[I 2025-12-07 15:20:53,067] Trial 49 finished with value: 0.20863309352517986 and parameters: {'n_estimators': 142, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 1, 'class_weight': 'balanced_subsample'}. Best is trial 39 with value: 0.2631578947368421.


In [22]:
best_params = study_rf_smote.best_params

smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

final_rf_smote = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_rf_smote.fit(X_train_res, y_train_res)

,n_estimators,206
,criterion,'gini'
,max_depth,25
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
y_pred_test = final_rf_smote.predict(X_test)
print(classification_report(y_test, y_pred_test))


              precision    recall  f1-score   support

           0       0.99      0.98      0.98      5376
           1       0.19      0.32      0.24        93

    accuracy                           0.97      5469
   macro avg       0.59      0.65      0.61      5469
weighted avg       0.97      0.97      0.97      5469



# XGBOOST CLASSIFIER

## XGBOOST + Hyperparameter Tuning

In [ ]:
def objective_xgb(trial, X_train, y_train, X_val, y_val):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),        
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 50),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)

    return f1_score(y_val, y_pred_val)

In [ ]:
study_xgb = optuna.create_study(
    study_name="xgb_class_opt",
    direction="maximize",
    storage="sqlite:///.../models/gb_class.db",
    load_if_exists=True
)

study_xgb.optimize(lambda trial: objective_xgb(trial, X_train, y_train, X_val, y_val), n_trials=50, show_progress_bar=True)

[I 2025-12-07 15:20:53,966] A new study created in RDB with name: xgb_class_opt
Best trial: 0. Best value: 0.267176:   2%|▏         | 1/50 [00:01<01:08,  1.41s/it]

[I 2025-12-07 15:20:55,375] Trial 0 finished with value: 0.26717557251908397 and parameters: {'learning_rate': 0.12635809447385093, 'max_depth': 10, 'n_estimators': 480, 'subsample': 0.7083475220610644, 'colsample_bytree': 0.9546246644165886, 'gamma': 2.0796519428272884, 'min_child_weight': 3, 'scale_pos_weight': 45.951508768263764}. Best is trial 0 with value: 0.26717557251908397.


Best trial: 0. Best value: 0.267176:   4%|▍         | 2/50 [00:01<00:35,  1.37it/s]

[I 2025-12-07 15:20:55,637] Trial 1 finished with value: 0.02040816326530612 and parameters: {'learning_rate': 0.19637384604421604, 'max_depth': 7, 'n_estimators': 562, 'subsample': 0.8621427648541132, 'colsample_bytree': 0.7427823632614743, 'gamma': 8.026746255123333, 'min_child_weight': 10, 'scale_pos_weight': 3.6716785576455995}. Best is trial 0 with value: 0.26717557251908397.


Best trial: 0. Best value: 0.267176:   6%|▌         | 3/50 [00:02<00:32,  1.45it/s]

...

[I 2025-12-07 15:21:34,977] Trial 47 finished with value: 0.32432432432432434 and parameters: {'learning_rate': 0.22335029642037874, 'max_depth': 10, 'n_estimators': 438, 'subsample': 0.6299192178027494, 'colsample_bytree': 0.6786114681684803, 'gamma': 5.486835309607679, 'min_child_weight': 2, 'scale_pos_weight': 5.137841454458313}. Best is trial 46 with value: 0.32894736842105265.


Best trial: 46. Best value: 0.328947:  98%|█████████▊| 49/50 [00:41<00:00,  2.57it/s]

[I 2025-12-07 15:21:35,293] Trial 48 finished with value: 0.176 and parameters: {'learning_rate': 0.22137497991097638, 'max_depth': 11, 'n_estimators': 443, 'subsample': 0.6343510630997351, 'colsample_bytree': 0.6608043224990365, 'gamma': 7.623773120886611, 'min_child_weight': 2, 'scale_pos_weight': 5.1606293507735925}. Best is trial 46 with value: 0.32894736842105265.


Best trial: 46. Best value: 0.328947: 100%|██████████| 50/50 [00:41<00:00,  1.20it/s]

[I 2025-12-07 15:21:35,515] Trial 49 finished with value: 0.0 and parameters: {'learning_rate': 0.26792934246653993, 'max_depth': 10, 'n_estimators': 435, 'subsample': 0.6193995786142743, 'colsample_bytree': 0.6781224478758816, 'gamma': 5.47501416292495, 'min_child_weight': 2, 'scale_pos_weight': 1.2524296125602135}. Best is trial 46 with value: 0.32894736842105265.


In [26]:
best_params = study_xgb.best_params

final_xgb = XGBClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist"
)

final_xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.683076418435748
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [27]:
y_pred_test = final_xgb.predict(X_test)
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5376
           1       0.23      0.19      0.21        93

    accuracy                           0.98      5469
   macro avg       0.61      0.59      0.60      5469
weighted avg       0.97      0.98      0.97      5469



In [28]:
probs = final_xgb.predict_proba(X_test)[:, 1]

In [29]:
thresholds = np.linspace(0.01, 0.99, 200)

results = []

best_threshold = 0
best_f1 = 0

for t in thresholds:
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds)

    results.append([t, precision, recall, f1])

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best F1 score: {best_f1:.4f}")


Best threshold: 0.360
Best F1 score: 0.2783


In [30]:
final_preds = (probs >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      5376
           1       0.23      0.34      0.28        93

    accuracy                           0.97      5469
   macro avg       0.61      0.66      0.63      5469
weighted avg       0.98      0.97      0.97      5469



# CATBOOST

In [31]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

df_cat = pd.read_csv(path_to_data / 'data_catboost.csv')
df_cat.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,AGE,EXPERIENCE,bad
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,NaN,2.0,32,12,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,NaN,2.0,32,12,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,58,3,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,8,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,8,0


In [ ]:
y = df_cat["bad"]
X = df_cat.drop(columns=["bad", "ID"])

In [ ]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
print(cat_features)

['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


In [ ]:
for col in cat_features:
    X[col] = X[col].fillna("missing").astype(str)

In [ ]:
cat_idx = [X.columns.get_loc(c) for c in cat_features]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
train_pool = Pool(X_train, y_train, cat_features=cat_idx)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_idx)

In [38]:
model_cb = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    border_count=128,
    auto_class_weights='Balanced',  
    verbose=200
)

In [39]:
%time model_cb.fit(train_pool, eval_set=test_pool)

0:	test: 0.4948695	best: 0.4948695 (0)	total: 75.1ms	remaining: 1m 52s
200:	test: 0.6444311	best: 0.6447997 (168)	total: 5.13s	remaining: 33.2s
400:	test: 0.6932938	best: 0.6932938 (400)	total: 8.34s	remaining: 22.9s
600:	test: 0.7064251	best: 0.7077588 (489)	total: 14.7s	remaining: 22.1s
800:	test: 0.7077985	best: 0.7083315 (770)	total: 18.5s	remaining: 16.1s
1000:	test: 0.7070001	best: 0.7083315 (770)	total: 21.8s	remaining: 10.9s
1200:	test: 0.7102707	best: 0.7106835 (1160)	total: 25s	remaining: 6.22s
1400:	test: 0.7113492	best: 0.7115284 (1395)	total: 28.3s	remaining: 2s
1499:	test: 0.7117564	best: 0.7126545 (1483)	total: 29.9s	remaining: 0us

bestTest = 0.7126545299
bestIteration = 1483

Shrink model to first 1484 iterations.
CPU times: user 2min 22s, sys: 17.6 s, total: 2min 39s
Wall time: 30.2 s


In [40]:
probs = model_cb.predict_proba(test_pool)[:, 1]
preds_default = (probs >= 0.5).astype(int)

In [41]:
print(classification_report(y_test, preds_default))

              precision    recall  f1-score   support

           0       0.99      0.96      0.98      7169
           1       0.16      0.44      0.24       123

    accuracy                           0.95      7292
   macro avg       0.58      0.70      0.61      7292
weighted avg       0.98      0.95      0.96      7292



## Catboost + threshold tuning

In [42]:
thresholds = np.linspace(0.1, 0.9, 200)
best_t = 0.5
best_f1 = 0

for t in thresholds:
    preds_t = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds_t)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

In [43]:
final_preds = (probs >= best_t).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      7169
           1       0.20      0.40      0.26       123

    accuracy                           0.96      7292
   macro avg       0.59      0.69      0.62      7292
weighted avg       0.98      0.96      0.97      7292



# LIGHTGBM CLASSIFIER

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [59]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,          
    is_unbalance=True,    
    random_state=42,
    n_jobs=-1,        
    importance_type='gain',  
    verbose=-1              
)

In [60]:
if 'ID' in df.columns: df = df.drop('ID', axis=1)
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols: df[col] = df[col].astype('category')

X = df.drop('bad', axis=1)
y = df['bad']

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [62]:
lgbm_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
probs = lgbm_model.predict_proba(X_test)[:, 1]

In [63]:
print(classification_report(y_test, lgbm_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      0.91      0.95      7169
           1       0.08      0.46      0.14       123

    accuracy                           0.90      7292
   macro avg       0.53      0.69      0.54      7292
weighted avg       0.97      0.90      0.93      7292



In [64]:
feature_importance = pd.DataFrame({'feature': X.columns,'importance': lgbm_model.feature_importances_}).sort_values('importance', ascending=False)
feature_importance.head(20)

,feature,importance
15,LOG_INCOME,129173.562169
13,AGE,127562.952875
14,EXPERIENCE,93681.896914
6,NAME_FAMILY_STATUS,28058.561237
4,NAME_INCOME_TYPE,26847.776467
12,CNT_FAM_MEMBERS,19129.876660
5,NAME_EDUCATION_TYPE,18380.298199
1,FLAG_OWN_CAR,17775.111566
3,CNT_CHILDREN,14980.912199
2,FLAG_OWN_REALTY,14840.206510


## LIGHTGBM + Threshold Tuning

In [65]:
thresholds = np.linspace(0.1, 0.9, 100)
best_threshold = 0.5
best_f1 = 0.0

In [66]:
for thresh in thresholds:
    y_pred_custom = (probs >= thresh).astype(int)
    score = f1_score(y_test, y_pred_custom)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

In [67]:
final_preds = (probs >= best_threshold).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7169
           1       0.18      0.30      0.22       123

    accuracy                           0.97      7292
   macro avg       0.58      0.64      0.60      7292
weighted avg       0.97      0.97      0.97      7292



In [68]:
df2 = df.copy()

if 'ID' in df2.columns:
    df2 = df2.drop('ID', axis=1)

for col in df2.select_dtypes(include=['object']).columns:
    df2[col] = df2[col].astype('category')

X = df2.drop('bad', axis=1)
y = df2['bad']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# LIGHTGBM + Hyperparameter Tuning

In [69]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        **params
    )

    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)

    return f1_score(y_val, y_pred_val)

In [70]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2025-12-07 15:28:52,032] A new study created in memory with name: no-name-696991b2-b838-4e7f-b317-d51e6162b4c4
Best trial: 0. Best value: 0.150943:   2%|▏         | 1/50 [00:00<00:48,  1.02it/s]

[I 2025-12-07 15:28:53,023] Trial 0 finished with value: 0.1509433962264151 and parameters: {'n_estimators': 594, 'learning_rate': 0.07903633771638861, 'num_leaves': 42, 'max_depth': 4, 'min_child_samples': 32, 'subsample': 0.6128548235754012, 'colsample_bytree': 0.7013949539615364, 'reg_alpha': 0.19607825312892052, 'reg_lambda': 0.17517782468774443}. Best is trial 0 with value: 0.1509433962264151.


Best trial: 1. Best value: 0.206897:   4%|▍         | 2/50 [00:02<01:07,  1.40s/it]

[I 2025-12-07 15:28:54,718] Trial 1 finished with value: 0.20689655172413793 and parameters: {'n_estimators': 275, 'learning_rate': 0.06148220930327306, 'num_leaves': 49, 'max_depth': 11, 'min_child_samples': 42, 'subsample': 0.8261904810345557, 'colsample_bytree': 0.727201901243136, 'reg_alpha': 0.35018182535509745, 'reg_lambda': 0.0712320190377407}. Best is trial 1 with value: 0.20689655172413793.


Best trial: 2. Best value: 0.378788:   6%|▌         | 3/50 [00:04<01:25,  1.81s/it]

...

[I 2025-12-07 15:31:12,710] Trial 47 finished with value: 0.40298507462686567 and parameters: {'n_estimators': 471, 'learning_rate': 0.17457107653233447, 'num_leaves': 67, 'max_depth': 9, 'min_child_samples': 16, 'subsample': 0.9411985514088572, 'colsample_bytree': 0.903339796395648, 'reg_alpha': 0.019321609774600988, 'reg_lambda': 0.32351827971996105}. Best is trial 21 with value: 0.40298507462686567.


Best trial: 21. Best value: 0.402985:  98%|█████████▊| 49/50 [02:23<00:03,  3.56s/it]

[I 2025-12-07 15:31:16,007] Trial 48 finished with value: 0.39097744360902253 and parameters: {'n_estimators': 689, 'learning_rate': 0.14193481553721954, 'num_leaves': 50, 'max_depth': 7, 'min_child_samples': 44, 'subsample': 0.8764597227142066, 'colsample_bytree': 0.9515703532551643, 'reg_alpha': 0.009865253308333452, 'reg_lambda': 0.16037288358025215}. Best is trial 21 with value: 0.40298507462686567.


Best trial: 21. Best value: 0.402985: 100%|██████████| 50/50 [02:26<00:00,  2.94s/it]

[I 2025-12-07 15:31:18,966] Trial 49 finished with value: 0.3787878787878788 and parameters: {'n_estimators': 393, 'learning_rate': 0.19040662787831492, 'num_leaves': 64, 'max_depth': 10, 'min_child_samples': 49, 'subsample': 0.8849777652140816, 'colsample_bytree': 0.8581900559105033, 'reg_alpha': 0.0489319922533663, 'reg_lambda': 0.06363034100656144}. Best is trial 21 with value: 0.40298507462686567.


In [71]:
best_params = study.best_params

model_best = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    **best_params
)

In [72]:
model_best.fit(X_train, y_train)
probs = model_best.predict_proba(X_test)[:, 1]

In [73]:
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7169
           1       0.18      0.30      0.22       123

    accuracy                           0.97      7292
   macro avg       0.58      0.64      0.60      7292
weighted avg       0.97      0.97      0.97      7292



In [74]:
feature_importance = pd.DataFrame({'feature': X.columns,'importance': model_best.feature_importances_}).sort_values('importance', ascending=False)
feature_importance.head(20)

,feature,importance
13,AGE,9891
15,LOG_INCOME,8663
14,EXPERIENCE,6071
6,NAME_FAMILY_STATUS,1914
4,NAME_INCOME_TYPE,1853
12,CNT_FAM_MEMBERS,1552
5,NAME_EDUCATION_TYPE,1372
3,CNT_CHILDREN,933
7,NAME_HOUSING_TYPE,833
2,FLAG_OWN_REALTY,832


# LIGHTGBM + Hyperparameter Tuning + Threshold Tuning

In [75]:
thresholds = np.linspace(0.1, 0.99, 200)
best_f1 = 0
best_t = 0.5

for t in thresholds:
    preds = (probs >= t).astype(int)
    score = f1_score(y_test, preds)
    if score > best_f1:
        best_f1 = score
        best_t = t

In [76]:
best_t

np.float64(0.18944723618090453)

In [77]:
final_preds = (probs >= best_t).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7169
           1       0.29      0.41      0.34       123

    accuracy                           0.97      7292
   macro avg       0.64      0.69      0.66      7292
weighted avg       0.98      0.97      0.98      7292



In [87]:
path_to_model = path_to_repo / 'models'
joblib.dump(model_best, path_to_model / "best_lgbm.pkl")

['/Users/arystankamalov/Documents/GitHub/emlyon_ml_project/models/best_lgbm.pkl']

In [79]:
feature_importance = pd.DataFrame({'feature': X.columns,'importance': model_best.feature_importances_}).sort_values('importance', ascending=False)
feature_importance.head(20)

,feature,importance
13,AGE,9891
15,LOG_INCOME,8663
14,EXPERIENCE,6071
6,NAME_FAMILY_STATUS,1914
4,NAME_INCOME_TYPE,1853
12,CNT_FAM_MEMBERS,1552
5,NAME_EDUCATION_TYPE,1372
3,CNT_CHILDREN,933
7,NAME_HOUSING_TYPE,833
2,FLAG_OWN_REALTY,832
